In [3]:
import os
from dotenv import load_dotenv
from params.paths import ROOT_DIR, DATA_DIR
from dbio.representative_db import connect_db, get_person_by_column, get_closest_person_by_name

load_dotenv()

RELEVANCE_DATA_DIR = os.path.join(DATA_DIR, "relevance_data")
SPEECH_FILE_PATH = os.path.join(RELEVANCE_DATA_DIR, "speech.tsv")
DISCUSSION_FILE = os.path.join(RELEVANCE_DATA_DIR, "discussion.tsv")
ISSUE_FILE = os.path.join(RELEVANCE_DATA_DIR, "issue.tsv")
AGGREGATE_FILE = os.path.join(RELEVANCE_DATA_DIR, "aggregate.tsv")
conn = connect_db(
	dbname="kokkaidoc",
	user="postgres",
	password=os.getenv("PSQL_DATABASE_PASSWORD"),
	host="localhost",
	port="5432"
)

with conn.cursor() as cur:
	rows = get_person_by_column(cur, "name_kanji", "河野太郎")
	if len(rows) == 1:
		person_id = rows[0].person_id
		print(person_id)

# investigate files
with open(SPEECH_FILE_PATH, "r") as f:
	first_line = f.readline()
	second_line = f.readline()
	third_line = f.readline()
	print("------------------------------------------")
	print("PATH",SPEECH_FILE_PATH)
	print("columns\n", first_line)
	print("examples\n", second_line)
	print("examples\n", third_line)
with open(DISCUSSION_FILE, "r") as f:
	first_line = f.readline()
	second_line = f.readline()
	third_line = f.readline()
	print("------------------------------------------")
	print("PATH",DISCUSSION_FILE)
	print("columns\n", first_line)
	print("examples\n", second_line)
	print("examples\n", third_line)
with open(ISSUE_FILE, "r") as f:
	first_line = f.readline()
	second_line = f.readline()
	third_line = f.readline()
	print("------------------------------------------")
	print("PATH",ISSUE_FILE)
	print("columns\n", first_line)
	print("examples\n", second_line)
	print("examples\n", third_line)

with open(AGGREGATE_FILE, "r") as f:
	first_line = f.readline()
	second_line = f.readline()
	third_line = f.readline()
	print("------------------------------------------")
	print("PATH",AGGREGATE_FILE)
	print("columns\n", first_line)
	print("examples\n", second_line)
	print("examples\n", third_line)


1483
------------------------------------------
PATH /root/projects/kokkai_analysis/data/params/../data/relevance_data/speech.tsv
columns
 speechID	speakerID	speakerGroup	speakerPosition	speech	previous_speechID	subsequent_speechID	discussionID

examples
 120105254X02720200522_003	松島みどり	自由民主党・無所属の会		"○松島みどり君　ただいま議題となりました法律案につきまして、法務委員会における審査の経過及び結果を御報告申し上げます。

examples
 　本案は、法律事務の国際化、専門化及び複雑多様化により的確に対応し、渉外的法律関係の一層の安定を図る等のため、所要の措置を講じようとするものであります。

------------------------------------------
PATH /root/projects/kokkai_analysis/data/params/../data/relevance_data/discussion.tsv
columns
 discussionID	issueID	summary	participants	tags	is_relevant	is_productive	initiator	quality_reason

examples
 120105254X02720200522_003	120105254X02720200522	法律案の内容が報告されました。国際化に対応するため、外国法事務弁護士の規定整備、職務経験要件の緩和、共同法人の設立を可能にする内容で、参議院可決後、本委員会で原案通り可決されました。	['松島みどり']	['該当なし']	True	True	松島みどり	議題である法律案の紹介と目的、内容の概要が報告されており、案件に関連し建設的な議論の始まりを示しているため。

examples
 120105254X02720200522_011	120105254X02720200522	公益通報者保護法の見直し案

## lets first process the low hanging fruit of processing the aggregate file into separate files and attaching person ids

In [12]:
import json
output_dir = os.path.join(RELEVANCE_DATA_DIR, "aggregate_processed")
os.makedirs(output_dir, exist_ok=True)


conn = connect_db(
	dbname="kokkaidoc",
	user="postgres",
	password=os.getenv("PSQL_DATABASE_PASSWORD"),
	host="localhost",
	port="5432"
)

speaker_id_conversion = {
	"福島みずほ": "福島瑞穂"
}

cur = conn.cursor()
processed_datas = {}

with open(AGGREGATE_FILE, "r") as f, open(os.path.join(RELEVANCE_DATA_DIR, "aggregate_processed.jsonl"), "w") as f_out:
	columns = f.readline().strip().split("\t")
	for line_number, line in enumerate(f):
		line_values = line.strip().split("\t")
		if len(line_values) != len(columns):
			raise ValueError(f"Line {line_number} has {len(line_values)} columns, expected {len(columns)}")
		
		processed_data = {k: v for k, v in zip(columns, line_values)}
		print(processed_data)
		speaker_id = processed_data["speakerID"]
		if speaker_id in speaker_id_conversion:
			speaker_id = speaker_id_conversion[speaker_id]
		person_row_from_db = get_person_by_column(cur, "name_kanji", speaker_id)
		if len(person_row_from_db) != 1:
			print(f"Found {len(person_row_from_db)} persons with name {speaker_id}, passing for now")
			continue
		person_id = person_row_from_db[0].person_id
		if person_id in processed_datas:
			processed_datas[person_id]["Total_Count"] += int(processed_data["Total_Count"])
			processed_datas[person_id]["R_True_P_True"] += int(processed_data["R_True_P_True"])
			processed_datas[person_id]["R_True_P_False"] += int(processed_data["R_True_P_False"])
			processed_datas[person_id]["R_False_P_True"] += int(processed_data["R_False_P_True"])
			processed_datas[person_id]["R_False_P_False"] += int(processed_data["R_False_P_False"])
			processed_datas[person_id]["prop_R_True_P_True"] = processed_datas[person_id]["R_True_P_True"] / processed_datas[person_id]["Total_Count"]
			processed_datas[person_id]["prop_R_True_P_False"] = processed_datas[person_id]["R_True_P_False"] / processed_datas[person_id]["Total_Count"]
			processed_datas[person_id]["prop_R_False_P_True"] = processed_datas[person_id]["R_False_P_True"] / processed_datas[person_id]["Total_Count"]
			processed_datas[person_id]["prop_R_False_P_False"] = processed_datas[person_id]["R_False_P_False"] / processed_datas[person_id]["Total_Count"]
			processed_datas[person_id]["prop_R_True_P_True"] = processed_datas[person_id]["R_True_P_True"] / processed_datas[person_id]["Total_Count"]
			continue

		processed_data["person_id"] = person_id
		total_count = int(processed_data["Total_Count"])
		prop_prd_rel = int(processed_data["R_True_P_True"]) / total_count
		prop_notprd_rel = int(processed_data["R_True_P_False"]) / total_count
		prop_prd_notrel = int(processed_data["R_False_P_True"]) / total_count
		prop_notprd_notrel = int(processed_data["R_False_P_False"]) / total_count
		processed_data["prop_R_True_P_True"] = prop_prd_rel
		processed_data["prop_R_True_P_False"] = prop_notprd_rel
		processed_data["prop_R_False_P_True"] = prop_prd_notrel
		processed_data["prop_R_False_P_False"] = prop_notprd_notrel
		processed_data["Total_Count"] = int(processed_data["Total_Count"])
		processed_data["R_True_P_True"] = int(processed_data["R_True_P_True"])
		processed_data["R_True_P_False"] = int(processed_data["R_True_P_False"])
		processed_data["R_False_P_True"] = int(processed_data["R_False_P_True"])
		processed_data["R_False_P_False"] = int(processed_data["R_False_P_False"])
		processed_datas[person_id] = processed_data

		# f_out.write(json.dumps(processed_data, ensure_ascii=False) + "\n") 
		# with open(os.path.join(output_dir, f"{person_id}.json"), "w") as f:
		# 	json.dump(processed_data, f, ensure_ascii=False)

	for person_id, data in processed_datas.items():
		f_out.write(json.dumps(data, ensure_ascii=False) + "\n")
		with open(os.path.join(output_dir, f"{person_id}.json"), "w") as f:
			json.dump(data, f, ensure_ascii=False)

cur.close()
conn.close()
# for how many people did we find data?
print(f"Found data for {len(os.listdir(output_dir))} people")


{'speakerID': '山添拓', 'speakerGroup': '日本共産党', 'R_True_P_True': '285', 'R_True_P_False': '242', 'R_False_P_True': '0', 'R_False_P_False': '18', 'Total_Count': '545'}
{'speakerID': '小西洋之', 'speakerGroup': '立憲民主・社民', 'R_True_P_True': '149', 'R_True_P_False': '276', 'R_False_P_True': '0', 'R_False_P_False': '40', 'Total_Count': '465'}
{'speakerID': '倉林明子', 'speakerGroup': '日本共産党', 'R_True_P_True': '283', 'R_True_P_False': '130', 'R_False_P_True': '0', 'R_False_P_False': '14', 'Total_Count': '427'}
{'speakerID': '東徹', 'speakerGroup': '日本維新の会', 'R_True_P_True': '261', 'R_True_P_False': '124', 'R_False_P_True': '0', 'R_False_P_False': '12', 'Total_Count': '397'}
{'speakerID': '川田龍平', 'speakerGroup': '立憲民主・社民', 'R_True_P_True': '281', 'R_True_P_False': '87', 'R_False_P_True': '0', 'R_False_P_False': '14', 'Total_Count': '382'}
{'speakerID': '田島麻衣子', 'speakerGroup': '立憲民主・社民', 'R_True_P_True': '232', 'R_True_P_False': '130', 'R_False_P_True': '0', 'R_False_P_False': '14', 'Total_Count': '376'}


## Now lets process the discussion file

In [9]:
from params.paths import DATA_DIR

repr_speeches_organized_dir = os.path.join(DATA_DIR, "repr_speeches_id_organized")
all_speeches_with_relevance_dir = os.path.join(DATA_DIR, "data_all_speeches_with_prd_and_rl")
os.makedirs(all_speeches_with_relevance_dir, exist_ok=True)

conn = connect_db(
	dbname="kokkaidoc",
	user="postgres",
	password=os.getenv("PSQL_DATABASE_PASSWORD"),
	host="localhost",
	port="5432"
)

def return_files_with_speech_id(speech_id:str, repr_speeches_dir:str):
	if not os.path.exists(repr_speeches_dir):
		return []
	repr_speeches_files = os.listdir(repr_speeches_dir)
	files_with_speech_id = []
	for file in repr_speeches_files:
		with open(os.path.join(repr_speeches_dir, file), "r") as f:
			for line in f:
				data = json.loads(line)
				if data['speechID'] == speech_id:
					files_with_speech_id.append(file)
	return files_with_speech_id

def create_new_file_with_productivity_flag_for_speech_id(
	speech_id: str,
	input_file_path: str,
	input_file_name: str,
	output_dir: str,
	is_productive: bool,
	is_relevant: bool,
	quality_reason: str,
):
	tmp_file_path = os.path.join(output_dir, f"tmp_{input_file_name}")
	# print("tmp file path", tmp_file_path)
	with open(input_file_path, "r") as f, open(tmp_file_path, "w") as f_out:
		for line in f:
			data = json.loads(line)
			if data['speechID'] == speech_id:
				data['is_productive'] = is_productive
				data['is_relevant'] = is_relevant
				data['quality_reason'] = quality_reason
			f_out.write(json.dumps(data, ensure_ascii=False) + "\n")
	os.remove(input_file_path)
	os.rename(tmp_file_path, os.path.join(output_dir, input_file_name))




cur = conn.cursor()

print("Processing discussion file")
with open(DISCUSSION_FILE, "r") as f:
	columns = f.readline().strip().split("\t")
	number_of_lines = sum(1 for line in f)
	f.seek(0)
	next(f)

	print("Number of lines", number_of_lines)
	for line_number, line in enumerate(f):
		print("Processing line number", line_number, "of", number_of_lines)
		line_values = line.strip().split("\t")
		if len(line_values) != len(columns):
			raise ValueError(f"Line {line_number} has {len(line_values)} columns, expected {len(columns)}")
		
		processed_data = {k: v for k, v in zip(columns, line_values)}
		# print("Processed data for line number", line_number, "of", number_of_lines, "\n", processed_data)
		# print(processed_data)
		initiator = processed_data['initiator']
		if initiator in speaker_id_conversion:
			initiator = speaker_id_conversion[initiator]
		quality_reason = processed_data['quality_reason']
		issue_id = processed_data['issueID']
		speech_id = processed_data['discussionID']
		is_productive = processed_data['is_productive']
		is_relevant = processed_data['is_relevant']

		person_row_from_db = get_person_by_column(cur, "name_kanji", initiator)
		if len(person_row_from_db) == 1:
			# print(f"Found person with name {initiator}, writing to the repr speeches files")
		
			person_id = person_row_from_db[0].person_id
			# print("found person id", person_id, "for", initiator, "with speech id", speech_id)

			# writing to the repr speeches files
			repr_speeches_dir = os.path.join(repr_speeches_organized_dir, str(person_id))
			files_with_speech_id = return_files_with_speech_id(speech_id, repr_speeches_dir)
			if len(files_with_speech_id) == 0:
				continue
			for file in files_with_speech_id:
				if "tmp_" in file:
					continue
				file_path = os.path.join(repr_speeches_dir, file)
				create_new_file_with_productivity_flag_for_speech_id(
					speech_id=speech_id,
					input_file_path=file_path,
					input_file_name=file,
					output_dir=repr_speeches_dir,
					is_productive=is_productive,
					is_relevant=is_relevant,
					quality_reason=quality_reason,
				)
		# now lets change the file we have in data_all_speeches_with_prd_and_rl
		issue_dir = os.path.join(all_speeches_with_relevance_dir, str(issue_id))
		speeches_file_path = os.path.join(issue_dir, "speeches.jsonl")
		output_tmp_file = os.path.join(issue_dir, "tmp_speeches.jsonl")
		with open(speeches_file_path, "r") as f, open(output_tmp_file, "w") as f_out:
			for line in f:
				data = json.loads(line)
				if data['speechID'] == speech_id:
					data['is_productive'] = is_productive
					data['is_relevant'] = is_relevant
					data['quality_reason'] = quality_reason
				f_out.write(json.dumps(data, ensure_ascii=False) + "\n")
		os.remove(speeches_file_path)
		os.rename(output_tmp_file, speeches_file_path)

cur.close()
conn.close()


Processing discussion file
Number of lines 28868
Processing line number 0 of 28868
Processing line number 1 of 28868
Processing line number 2 of 28868
Processing line number 3 of 28868
Processing line number 4 of 28868
Processing line number 5 of 28868
Processing line number 6 of 28868
Processing line number 7 of 28868
Processing line number 8 of 28868
Processing line number 9 of 28868
Processing line number 10 of 28868
Processing line number 11 of 28868
Processing line number 12 of 28868
Processing line number 13 of 28868
Processing line number 14 of 28868
Processing line number 15 of 28868
Processing line number 16 of 28868
Processing line number 17 of 28868
Processing line number 18 of 28868
Processing line number 19 of 28868
Processing line number 20 of 28868
Processing line number 21 of 28868
Processing line number 22 of 28868
Processing line number 23 of 28868
Processing line number 24 of 28868
Processing line number 25 of 28868
Processing line number 26 of 28868
Processing line 

KeyboardInterrupt: 